# ARIA — Entrenar un adaptador LoRA (gratis, en la nube)

Este cuaderno entrena un **adaptador LoRA** (pequeño, menos de 300 MB) para el modelo ARIA,
usando el archivo `aria-chat.jsonl` que genera el repositorio con:

```
npm run db:dataset
```

- **No usa tu computador**: corre en la GPU gratuita de Kaggle (o Colab).
- **No sube tus datos a ningún lado público**: el archivo se queda en tu sesión privada.
- **Formato del archivo**: una línea JSON por conversación: `{"messages":[{"role":...},...]}`.

> Aviso honesto: este cuaderno es un **punto de partida**. Las librerías (unsloth, trl, peft)
cambian de versión seguido. Si alguna celda falla por un nombre de parámetro, mirá las notas
en rojo de cada sección y los enlaces del final.

## 1. Instalar las librerías

En Kaggle: activá **Internet** en la barra lateral derecha antes de ejecutar esta celda.

In [ ]:
# Instalación de la pila de entrenamiento (QLoRA).
!pip install -q -U unsloth
!pip install -q -U trl peft accelerate bitsandbytes datasets

## 2. Subir el archivo de datos

Tenés dos opciones:

1. **Recomendada**: subí `aria-chat.jsonl` como un *dataset* **privado** de Kaggle y adjuntalo a este cuaderno.
2. Arrastrá `aria-chat.jsonl` a la carpeta de trabajo (queda en `/kaggle/working/`).

Ajustá `RUTA_DATOS` abajo según dónde te haya quedado.

**Recordá**: este archivo tiene tus conversaciones. Mantenelo privado.

In [ ]:
from pathlib import Path

# Cambiá esta ruta si subiste el archivo a otro lugar.
RUTA_DATOS = Path("/kaggle/input/aria-dataset/aria-chat.jsonl")
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path("/kaggle/working/aria-chat.jsonl")

print("Archivo:", RUTA_DATOS)
print("¿Existe?:", RUTA_DATOS.exists())

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=str(RUTA_DATOS), split="train")
print("Ejemplos:", len(ds))
print("Roles del primero:", [m["role"] for m in ds[0]["messages"]])

## 3. Cargar el modelo base

Usamos la **misma familia** que ofrece Cloudflare (`Qwen2.5-Coder-7B-Instruct`) para que el
adaptador encaje después. Se carga en **4 bits** para que entre en una GPU de 16 GB.

- En Kaggle: menú *Settings* → *Accelerator* → **GPU T4 x2** (o P100).
- Si tu conjunto de datos es muy chico (pocos ejemplos), el modelo va a *memorizar*. Sirve para
  **probar que el circuito funciona**, no para tener un buen modelo todavía.

In [ ]:
from unsloth import FastLanguageModel

MODELO_BASE = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
MAX_LARGO = 1024  # longitud máxima de cada ejemplo, en "tokens"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO_BASE,
    max_seq_length=MAX_LARGO,
    load_in_4bit=True,
)
print("Modelo cargado:", MODELO_BASE)

## 4. Configurar el adaptador LoRA

Usamos **rango 8** (`r=8`) a propósito: las LoRA de Cloudflare solo aceptan rango **<= 8**, y el
archivo final queda bien por debajo de los 300 MB permitidos.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # Cloudflare exige rango <= 8
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Adaptador configurado (r=8).")

## 5. Entrenar

Convertimos cada conversación a texto con la *plantilla de chat* del modelo y entrenamos.

> Nota de versión: en algunas versiones de `trl`, `dataset_text_field` y `max_seq_length` van
dentro de `SFTConfig`; en otras, como argumentos de `SFTTrainer`. Si te da error de parámetro
inesperado, mové esos dos al otro lugar.

In [ ]:
from trl import SFTTrainer, SFTConfig


def a_texto(ejemplo):
    return {
        "text": tokenizer.apply_chat_template(
            ejemplo["messages"], tokenize=False, add_generation_prompt=False
        )
    }


ds_texto = ds.map(a_texto)

entrenador = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_texto,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LARGO,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="salida",
        report_to="none",
    ),
)
entrenador.train()

## 6. Guardar el adaptador

Se guarda en la carpeta `aria-lora/`, con el `adapter_config.json` y el `adapter_model.safetensors`
que pide Cloudflare. Después lo bajás comprimido.

In [ ]:
import shutil

model.save_pretrained("aria-lora")
tokenizer.save_pretrained("aria-lora")

shutil.make_archive("aria-lora", "zip", "aria-lora")
print("Listo. Archivos dentro de aria-lora/")
print("Comprimido: aria-lora.zip (bajalo desde el panel de archivos)")

## Siguientes pasos

1. **Bajá** `aria-lora.zip` desde el panel *Output* de Kaggle.
2. **No lo subas** a ningún repositorio público de Hugging Face (regla del proyecto).
3. Para **usarlo en ARIA**: hoy la familia de bases **LoRA de Cloudflare devuelve error 500**,
   incluso sin adaptador. Mientras eso no se resuelva, el adaptador queda guardado para cuando
   funcione.
4. Cuando tengas **muchas más conversaciones**, volvé a generar el conjunto con
   `npm run db:dataset` y repetí este cuaderno.

### Enlaces

- Unsloth: https://github.com/unslothai/unsloth
- TRL / SFTTrainer: https://huggingface.co/docs/trl
- LoRA en Cloudflare: https://developers.cloudflare.com/workers-ai/fine-tunes/